# FIBO + Neo4j — Exploratory Data Analysis

Hands-on EDA of the FIBO ontology (T-Box) and the seeded counterparty/LEI graph (A-Box).

**Prereqs**: run the Cypher files in order before this notebook.

1. `cypher/00_setup.cypher`
2. `cypher/10_load_fibo.cypher`
3. `cypher/25_seed_abox.cypher`
4. `cypher/40_gds_analytics.cypher` (writes `systemicScore` and `communityId`)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from dotenv import load_dotenv
from neo4j import GraphDatabase

sns.set_theme(style='whitegrid')
load_dotenv()
URI  = os.getenv('NEO4J_URI', 'neo4j://127.0.0.1:7687')
USER = os.getenv('NEO4J_USER', 'neo4j')
PWD  = os.getenv('NEO4J_PASSWORD', '12345678')
DB   = os.getenv('NEO4J_DATABASE', 'neo4j')
driver = GraphDatabase.driver(URI, auth=(USER, PWD))

def q(cypher, **params):
    with driver.session(database=DB) as s:
        return pd.DataFrame([r.data() for r in s.run(cypher, **params)])

q('RETURN 1 AS ok')

## 1. Inventory — labels & relationship types

In [ ]:
labels = q('CALL db.labels() YIELD label CALL { WITH label MATCH (n) WHERE label IN labels(n) RETURN count(n) AS c } RETURN label, c ORDER BY c DESC')
labels

In [ ]:
rels = q("""
MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS c ORDER BY c DESC
""")
rels.head(20)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
labels.head(15).plot.barh(x='label', y='c', ax=ax[0], color='steelblue', legend=False).invert_yaxis()
ax[0].set_title('Top labels by node count')
rels.head(15).plot.barh(x='rel', y='c', ax=ax[1], color='seagreen', legend=False).invert_yaxis()
ax[1].set_title('Top relationship types')
plt.tight_layout(); plt.show()

## 2. T-Box — FIBO class hierarchy depth

In [ ]:
tbox = q("""
MATCH path = (leaf:Class)-[:SCO*]->(root:Class)
WHERE NOT (leaf)<-[:SCO]-(:Class) AND NOT (root)-[:SCO]->(:Class)
RETURN length(path) AS depth
""")
if not tbox.empty:
    print('Hierarchy depth — min/median/max:', tbox.depth.min(), tbox.depth.median(), tbox.depth.max())
    sns.histplot(tbox.depth, bins=range(1, tbox.depth.max()+2)); plt.title('FIBO class hierarchy depth'); plt.show()
else:
    print('No :Class nodes — load the FIBO ontology first.')

In [ ]:
# Subclass tree of LegalEntity
le = q("""
MATCH path = (sub:Class)-[:SCO*1..6]->(le:Class)
WHERE le.uri ENDS WITH 'LegalEntity'
RETURN sub.label AS subclass, length(path) AS depth ORDER BY depth, subclass
""")
le.head(40)

## 3. A-Box — counterparty network EDA

In [ ]:
deg = q("""
MATCH (e:LegalEntity)
OPTIONAL MATCH (e)-[r_out]->(:LegalEntity)
OPTIONAL MATCH (e)<-[r_in]-(:LegalEntity)
RETURN e.name AS entity, e.jurisdictionCode AS country, e.sectorName AS sector,
       count(DISTINCT r_out) AS out_deg, count(DISTINCT r_in) AS in_deg
ORDER BY (count(DISTINCT r_out)+count(DISTINCT r_in)) DESC
""")
deg

In [ ]:
exposure = q("""
MATCH (b:Bank)-[l:LENDS_TO]->(borrower:LegalEntity)
OPTIONAL MATCH path=(borrower)-[:HAS_CONTROLLING_PARTY*0..5]->(top)
WHERE NOT (top)-[:HAS_CONTROLLING_PARTY]->()
WITH b, top, sum(l.exposure_musd) AS exposure
RETURN b.name AS bank, coalesce(top.name,'INDEPENDENT') AS control_group, exposure
ORDER BY bank, exposure DESC
""")
exposure

In [ ]:
if not exposure.empty:
    pivot = exposure.pivot_table(index='control_group', columns='bank', values='exposure', fill_value=0)
    pivot.plot(kind='bar', stacked=True, figsize=(10,5), colormap='tab20')
    plt.ylabel('Exposure (M USD)'); plt.title('Bank exposure by ultimate control group'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## 4. Pull the network into NetworkX for centrality cross-checks

In [ ]:
edges = q("""
MATCH (a:LegalEntity)-[r:HAS_OWNERSHIP_IN]->(b:LegalEntity)
RETURN a.name AS src, b.name AS dst, r.weight AS w
""")
G = nx.DiGraph()
for _, e in edges.iterrows():
    G.add_edge(e.src, e.dst, weight=float(e.w or 1.0))
print(G.number_of_nodes(), 'nodes /', G.number_of_edges(), 'edges')
pr = nx.pagerank(G, weight='weight')
pd.Series(pr).sort_values(ascending=False).head(10).plot.barh(color='indianred')
plt.gca().invert_yaxis(); plt.title('NetworkX PageRank (ownership graph)'); plt.tight_layout(); plt.show()

## 5. GDS results (must have run `40_gds_analytics.cypher` first)

In [ ]:
gds = q("""
MATCH (e:LegalEntity)
RETURN e.name AS entity, e.jurisdictionCode AS country, e.sectorName AS sector,
       e.systemicScore AS pagerank, e.communityId AS community
ORDER BY pagerank DESC
""")
gds

In [ ]:
if 'pagerank' in gds and gds.pagerank.notna().any():
    sns.scatterplot(data=gds, x='pagerank', y='entity', hue='community', palette='tab10', s=120)
    plt.title('Systemic importance vs. Louvain community'); plt.tight_layout(); plt.show()

## 6. Default-cascade simulation (Apex Holdings)

In [ ]:
cascade = q("""
MATCH (root:LegalEntity {name:'Apex Holdings Inc.'})
MATCH (root)-[:HAS_OWNERSHIP_IN*1..5]->(d)
OPTIONAL MATCH (b:Bank)-[l:LENDS_TO]->(d)
RETURN d.name AS at_risk, collect(DISTINCT b.name) AS banks, sum(l.exposure_musd) AS exposure
ORDER BY exposure DESC
""")
cascade

In [ ]:
driver.close()